# **04 - Train Sub-Models (With Weighted Loss for Class Imbalance)**
Melatih 2 specialized models:
- Organic Branch: MobileNetV3 (2 classes)
- Inorganic Branch: ConvNeXt-Tiny (7 classes)

**Key Improvement:** Menggunakan weighted loss untuk handle class imbalance di setiap branch

## **Import Library**

In [ ]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"GPU Available: {torch.cuda.is_available()}")

# Paths
BASE_SUB_DIR = "../data/processed/level_2_sub"
SAVED_MODELS_DIR = "../saved_models"
RESULTS_DIR = "../results"

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print("\n✓ Setup selesai!")

## **Helper Functions**

In [ ]:
def calculate_class_weights(dataset, device):
    """
    Calculate balanced class weights untuk dataset
    """
    labels = np.array([label for _, label in dataset.samples])
    
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(labels),
        y=labels
    )
    
    return torch.FloatTensor(class_weights).to(device), labels, class_weights

def train_model(model, dataloaders, criterion, optimizer, num_epochs, class_names):
    """
    Train model dengan validation
    """
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    
    training_history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }
    
    for epoch in range(num_epochs):
        print(f'\nEpoch {epoch+1}/{num_epochs}')
        print('-' * 50)
        
        # Training
        model.train()
        train_loss = 0.0
        train_corrects = 0
        train_samples = 0
        
        for inputs, labels in dataloaders['train']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            
            with torch.set_grad_enabled(True):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_corrects += torch.sum(preds == labels.data)
            train_samples += inputs.size(0)
        
        train_loss_avg = train_loss / train_samples
        train_acc = train_corrects.double() / train_samples
        
        # Validation
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        val_samples = 0
        
        for inputs, labels in dataloaders['val']:
            inputs = inputs.to(device)
            labels = labels.to(device)
            
            with torch.set_grad_enabled(False):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            val_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            val_corrects += torch.sum(preds == labels.data)
            val_samples += inputs.size(0)
        
        val_loss_avg = val_loss / val_samples
        val_acc = val_corrects.double() / val_samples
        
        training_history['train_loss'].append(train_loss_avg)
        training_history['train_acc'].append(train_acc.item())
        training_history['val_loss'].append(val_loss_avg)
        training_history['val_acc'].append(val_acc.item())
        
        print(f'Train Loss: {train_loss_avg:.4f} | Train Acc: {train_acc:.4f}')
        print(f'Val Loss: {val_loss_avg:.4f} | Val Acc: {val_acc:.4f}')
        
        if val_acc > best_acc:
            best_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            print(f'→ Best model updated! (Acc: {best_acc:.4f})')
    
    model.load_state_dict(best_model_wts)
    time_elapsed = time.time() - since
    print(f'\n✓ Training complete!\n  Time: {int(time_elapsed)}s')
    
    return model, training_history

print("✓ Helper functions ready!")

## **Data Augmentation**

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.3),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ]),
    'test': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
}

print("✓ Data transforms ready!")

---
# **ORGANIC BRANCH - MobileNetV3 (2 Classes)**
---

## **Load Organic Data**

In [ ]:
print("\n" + "="*60)
print("ORGANIC BRANCH - MobileNetV3")
print("="*60)

org_dir = os.path.join(BASE_SUB_DIR, "organic_branch")

train_dataset_org = datasets.ImageFolder(
    os.path.join(org_dir, "train"),
    data_transforms['train']
)
val_dataset_org = datasets.ImageFolder(
    os.path.join(org_dir, "val"),
    data_transforms['val']
)
test_dataset_org = datasets.ImageFolder(
    os.path.join(org_dir, "test"),
    data_transforms['test']
)

dataloaders_org = {
    'train': DataLoader(train_dataset_org, batch_size=32, shuffle=True, num_workers=4),
    'val': DataLoader(val_dataset_org, batch_size=32, shuffle=False, num_workers=4),
    'test': DataLoader(test_dataset_org, batch_size=32, shuffle=False, num_workers=4)
}

class_names_org = train_dataset_org.classes
num_classes_org = len(class_names_org)

print(f"\nClasses: {class_names_org}")
print(f"Training: {len(train_dataset_org)} | Val: {len(val_dataset_org)} | Test: {len(test_dataset_org)}")

# Calculate class weights
weights_tensor_org, labels_org, weights_values_org = calculate_class_weights(train_dataset_org, device)

print(f"\nClass Distribution:")
for i, (class_name, count) in enumerate(zip(class_names_org, np.bincount(labels_org))):
    print(f"  {class_name}: {count} images (weight: {weights_values_org[i]:.4f})")

## **Build Organic Model**

In [ ]:
print("\n→ Loading pre-trained MobileNetV3...")

model_organic = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.DEFAULT)
model_organic.classifier[-1] = nn.Linear(model_organic.classifier[-1].in_features, num_classes_org)
model_organic = model_organic.to(device)

print(f"✓ Model loaded!")
print(f"  Total params: {sum(p.numel() for p in model_organic.parameters()):,}")

## **Loss & Optimizer (Organic)**

In [ ]:
criterion_org = nn.CrossEntropyLoss(weight=weights_tensor_org)
optimizer_org = optim.Adam(model_organic.parameters(), lr=1e-4, weight_decay=1e-4)

print("✓ Loss & Optimizer ready (Organic)")
print(f"  Loss: CrossEntropyLoss with weights")
print(f"  Optimizer: Adam (lr=1e-4)")

## **Train Organic Model**

In [ ]:
print("\n" + "="*60)
print("TRAINING ORGANIC MODEL")
print("="*60)

EPOCHS_ORG = 10
model_organic, history_org = train_model(
    model_organic,
    dataloaders_org,
    criterion_org,
    optimizer_org,
    num_epochs=EPOCHS_ORG,
    class_names=class_names_org
)

## **Save Organic Model**

In [ ]:
model_organic.load_state_dict(copy.deepcopy(model_organic.state_dict()))
org_model_path = os.path.join(SAVED_MODELS_DIR, 'sub_organic_best.pth')
torch.save(model_organic.state_dict(), org_model_path)

print(f"\n✓ Organic model saved: {org_model_path}")

---
# **INORGANIC BRANCH - ConvNeXt-Tiny (7 Classes)**
---

## **Load Inorganic Data**

In [ ]:
print("\n" + "="*60)
print("INORGANIC BRANCH - ConvNeXt-Tiny")
print("="*60)

inorg_dir = os.path.join(BASE_SUB_DIR, "inorganic_branch")

train_dataset_inorg = datasets.ImageFolder(
    os.path.join(inorg_dir, "train"),
    data_transforms['train']
)
val_dataset_inorg = datasets.ImageFolder(
    os.path.join(inorg_dir, "val"),
    data_transforms['val']
)
test_dataset_inorg = datasets.ImageFolder(
    os.path.join(inorg_dir, "test"),
    data_transforms['test']
)

dataloaders_inorg = {
    'train': DataLoader(train_dataset_inorg, batch_size=32, shuffle=True, num_workers=4),
    'val': DataLoader(val_dataset_inorg, batch_size=32, shuffle=False, num_workers=4),
    'test': DataLoader(test_dataset_inorg, batch_size=32, shuffle=False, num_workers=4)
}

class_names_inorg = train_dataset_inorg.classes
num_classes_inorg = len(class_names_inorg)

print(f"\nClasses: {class_names_inorg}")
print(f"Training: {len(train_dataset_inorg)} | Val: {len(val_dataset_inorg)} | Test: {len(test_dataset_inorg)}")

# Calculate class weights
weights_tensor_inorg, labels_inorg, weights_values_inorg = calculate_class_weights(train_dataset_inorg, device)

print(f"\nClass Distribution:")
for i, (class_name, count) in enumerate(zip(class_names_inorg, np.bincount(labels_inorg))):
    print(f"  {class_name}: {count} images (weight: {weights_values_inorg[i]:.4f})")

## **Build Inorganic Model**

In [ ]:
print("\n→ Loading pre-trained ConvNeXt-Tiny...")

model_inorganic = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
model_inorganic.classifier[-1] = nn.Linear(model_inorganic.classifier[-1].in_features, num_classes_inorg)
model_inorganic = model_inorganic.to(device)

print(f"✓ Model loaded!")
print(f"  Total params: {sum(p.numel() for p in model_inorganic.parameters()):,}")

## **Loss & Optimizer (Inorganic)**

In [ ]:
criterion_inorg = nn.CrossEntropyLoss(weight=weights_tensor_inorg)
optimizer_inorg = optim.Adam(model_inorganic.parameters(), lr=5e-5, weight_decay=1e-4)

print("✓ Loss & Optimizer ready (Inorganic)")
print(f"  Loss: CrossEntropyLoss with weights")
print(f"  Optimizer: Adam (lr=5e-5)")

## **Train Inorganic Model**

In [ ]:
print("\n" + "="*60)
print("TRAINING INORGANIC MODEL")
print("="*60)

EPOCHS_INORG = 10
model_inorganic, history_inorg = train_model(
    model_inorganic,
    dataloaders_inorg,
    criterion_inorg,
    optimizer_inorg,
    num_epochs=EPOCHS_INORG,
    class_names=class_names_inorg
)

## **Save Inorganic Model**

In [ ]:
model_inorganic.load_state_dict(copy.deepcopy(model_inorganic.state_dict()))
inorg_model_path = os.path.join(SAVED_MODELS_DIR, 'sub_inorganic_best.pth')
torch.save(model_inorganic.state_dict(), inorg_model_path)

print(f"\n✓ Inorganic model saved: {inorg_model_path}")

---
# **EVALUATION**
---

## **Evaluate Organic Model**

In [ ]:
print("\n" + "="*60)
print("EVALUATING ORGANIC MODEL")
print("="*60)

model_organic.eval()
all_preds_org = []
all_labels_org = []

with torch.no_grad():
    for inputs, labels in dataloaders_org['test']:
        inputs = inputs.to(device)
        outputs = model_organic(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds_org.extend(preds.cpu().numpy())
        all_labels_org.extend(labels.numpy())

all_preds_org = np.array(all_preds_org)
all_labels_org = np.array(all_labels_org)

test_acc_org = np.mean(all_preds_org == all_labels_org)
print(f"\nTest Accuracy: {test_acc_org:.4f}")
print(f"\nClassification Report:")
print(classification_report(all_labels_org, all_preds_org, target_names=class_names_org, digits=4))

## **Evaluate Inorganic Model**

In [ ]:
print("\n" + "="*60)
print("EVALUATING INORGANIC MODEL")
print("="*60)

model_inorganic.eval()
all_preds_inorg = []
all_labels_inorg = []

with torch.no_grad():
    for inputs, labels in dataloaders_inorg['test']:
        inputs = inputs.to(device)
        outputs = model_inorganic(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds_inorg.extend(preds.cpu().numpy())
        all_labels_inorg.extend(labels.numpy())

all_preds_inorg = np.array(all_preds_inorg)
all_labels_inorg = np.array(all_labels_inorg)

test_acc_inorg = np.mean(all_preds_inorg == all_labels_inorg)
print(f"\nTest Accuracy: {test_acc_inorg:.4f}")
print(f"\nClassification Report:")
print(classification_report(all_labels_inorg, all_preds_inorg, target_names=class_names_inorg, digits=4))

## **Summary**

In [ ]:
print("\n" + "="*60)
print("SUB-MODELS TRAINING SUMMARY")
print("="*60)

print(f"\nORGANIC MODEL (MobileNetV3):")
print(f"  Final Train Acc: {history_org['train_acc'][-1]:.4f}")
print(f"  Final Val Acc: {history_org['val_acc'][-1]:.4f}")
print(f"  Test Acc: {test_acc_org:.4f}")
print(f"  Saved: {org_model_path}")

print(f"\nINORGANIC MODEL (ConvNeXt-Tiny):")
print(f"  Final Train Acc: {history_inorg['train_acc'][-1]:.4f}")
print(f"  Final Val Acc: {history_inorg['val_acc'][-1]:.4f}")
print(f"  Test Acc: {test_acc_inorg:.4f}")
print(f"  Saved: {inorg_model_path}")

print("\n" + "="*60)
print("✓ Both sub-models trained successfully!")
print("="*60)